# Experiment 03 — Clean Local-Scale CQR Prototype

This is a clean reconstruction of the historical 1,000-observation prototype. It evaluates empirical chronological-block coverage; ordinary split-conformal iid/exchangeability guarantees do not apply to these dependent, regime-shifting time-series blocks.

## 1. Experiment Configuration

In [31]:
SEED = 42
N = 1000
SEQUENCE_LENGTH = 5
TRAIN_END = 600
SCALE_END = 750
CALIBRATION_END = 850
ALPHA = 0.10
Q_LOW = 0.10
Q_HIGH = 0.90
HIDDEN_SIZE = 32
NUM_LAYERS = 1
EPOCHS = 40
LEARNING_RATE = 1e-3
LOW_SIGMA = 0.2
HIGH_SIGMA = 1.0
EXPECTED_CALIBRATION_SIZE = 95
EXPECTED_CONFORMAL_RANK = 87
ORACLE_HIGH_LOW_SCALE_RATIO = HIGH_SIGMA / LOW_SIGMA

print({name: value for name, value in globals().items() if name.isupper()})

{'SEED': 42, 'N': 1000, 'SEQUENCE_LENGTH': 5, 'TRAIN_END': 600, 'SCALE_END': 750, 'CALIBRATION_END': 850, 'ALPHA': 0.1, 'Q_LOW': 0.1, 'Q_HIGH': 0.9, 'HIDDEN_SIZE': 32, 'NUM_LAYERS': 1, 'EPOCHS': 40, 'LEARNING_RATE': 0.001, 'LOW_SIGMA': 0.2, 'HIGH_SIGMA': 1.0, 'EXPECTED_CALIBRATION_SIZE': 95, 'EXPECTED_CONFORMAL_RANK': 87, 'ORACLE_HIGH_LOW_SCALE_RATIO': 5.0}


## 2. Imports and Reproducibility

In [32]:
import platform
import random
import sys
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cpu')

print(f'Python: {sys.version.split()[0]}')
print(f'NumPy: {np.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'OS: {platform.platform()}')
print(f'Device: {DEVICE}')
print(f'Seed: {SEED}')

Python: 3.13.15
NumPy: 2.1.3
PyTorch: 2.11.0+cpu
OS: Linux-6.6.122+-x86_64-with-glibc2.35
Device: cpu
Seed: 42


## 3. Data-Generating Process

In [33]:
t = np.arange(N)
signal = np.sin(2 * np.pi * t / 50)
true_sigma = np.where((t // 100) % 2 == 0, LOW_SIGMA, HIGH_SIGMA)
y = signal + true_sigma * np.random.normal(0.0, 1.0, N)
regime_label = np.where(true_sigma == LOW_SIGMA, 'low', 'high')  # evaluation only

assert y.shape == true_sigma.shape == regime_label.shape == (N,)
assert np.sum(regime_label == 'low') == 500
assert np.sum(regime_label == 'high') == 500
print('DGP: y_t = sin(2πt/50) + sigma_t ε_t; alternating 100-point low/high blocks.')

DGP: y_t = sin(2πt/50) + sigma_t ε_t; alternating 100-point low/high blocks.


## 4. Chronological Splits

In [34]:
raw_split_indices = {
    'train': np.arange(0, TRAIN_END),
    'scale': np.arange(TRAIN_END, SCALE_END),
    'calibration': np.arange(SCALE_END, CALIBRATION_END),
    'test': np.arange(CALIBRATION_END, N),
}
for name, indices in raw_split_indices.items():
    print(f'{name:12s}: raw indices {indices[0]}–{indices[-1]}, n={len(indices)}')
assert [len(v) for v in raw_split_indices.values()] == [600, 150, 100, 150]

train       : raw indices 0–599, n=600
scale       : raw indices 600–749, n=150
calibration : raw indices 750–849, n=100
test        : raw indices 850–999, n=150


## 5. Sequence Construction

In [35]:
def create_split_sequences(values, split_indices, sequence_length):
    """Construct sequences inside one chronological split only.

    Each target y_t uses y_(t-L),...,y_(t-1), all historical observations.
    The first L raw observations in each split are intentionally not targets.
    """
    X, targets, target_indices = [], [], []
    first_index = split_indices[0]
    for target_index in split_indices:
        if target_index - sequence_length < first_index:
            continue
        X.append(values[target_index-sequence_length:target_index])
        targets.append(values[target_index])
        target_indices.append(target_index)
    return np.asarray(X), np.asarray(targets), np.asarray(target_indices)

split_data = {}
for name, indices in raw_split_indices.items():
    X, target, target_indices = create_split_sequences(y, indices, SEQUENCE_LENGTH)
    split_data[name] = {'X': X, 'y': target, 'target_indices': target_indices}
    print(f'{name:12s}: X={X.shape}, target range={target_indices[0]}–{target_indices[-1]}')

assert split_data['train']['X'].shape == (595, 5)
assert split_data['scale']['X'].shape == (145, 5)
assert split_data['calibration']['X'].shape == (95, 5)
assert split_data['test']['X'].shape == (145, 5)
assert np.max(split_data['calibration']['target_indices']) < np.min(split_data['test']['target_indices'])

def as_tensor(array):
    return torch.as_tensor(array, dtype=torch.float32, device=DEVICE)

for values in split_data.values():
    values['X_tensor'] = as_tensor(values['X']).unsqueeze(-1)
    values['y_tensor'] = as_tensor(values['y'])

train       : X=(595, 5), target range=5–599
scale       : X=(145, 5), target range=605–749
calibration : X=(95, 5), target range=755–849
test        : X=(145, 5), target range=855–999


## 6. Quantile LSTM

In [36]:
class QuantileLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=HIDDEN_SIZE, num_layers=NUM_LAYERS):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.head = nn.Linear(hidden_size, 2)

    def forward(self, x):
        output, _ = self.lstm(x)
        return self.head(output[:, -1, :])

def pinball_loss(prediction, target, quantiles=(Q_LOW, Q_HIGH)):
    losses = []
    for column, quantile in enumerate(quantiles):
        error = target - prediction[:, column]
        losses.append(torch.maximum((quantile - 1) * error, quantile * error))
    return torch.stack(losses, dim=1).mean()

torch.manual_seed(SEED)
model = QuantileLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    train_prediction = model(split_data['train']['X_tensor'])
    loss = pinball_loss(train_prediction, split_data['train']['y_tensor'])
    loss.backward()
    optimizer.step()

print(f'Trained QuantileLSTM for {EPOCHS} full-batch epochs; final train pinball loss={loss.item():.6f}.')

Trained QuantileLSTM for 40 full-batch epochs; final train pinball loss=0.252745.


## 7. Base Quantile Predictions

In [37]:
@torch.no_grad()
def predict_quantiles(X_tensor):
    model.eval()
    prediction = model(X_tensor)
    return prediction[:, 0].cpu().numpy(), prediction[:, 1].cpu().numpy()

for name in ('scale', 'calibration', 'test'):
    q_low, q_high = predict_quantiles(split_data[name]['X_tensor'])
    split_data[name]['q_low'] = q_low
    split_data[name]['q_high'] = q_high
    crossings = int(np.sum(q_low > q_high))
    print(f'{name:12s}: predictions={q_low.shape}, quantile crossings={crossings}')
    assert q_low.shape == q_high.shape == split_data[name]['y'].shape
    assert np.isfinite(q_low).all() and np.isfinite(q_high).all()
    assert crossings == 0

cal_y = split_data['calibration']['y']
cal_q_low, cal_q_high = split_data['calibration']['q_low'], split_data['calibration']['q_high']
test_y = split_data['test']['y']
test_q_low, test_q_high = split_data['test']['q_low'], split_data['test']['q_high']
test_sigma = true_sigma[split_data['test']['target_indices']]  # evaluation/oracle only
cal_sigma = true_sigma[split_data['calibration']['target_indices']]  # oracle only
test_regime = regime_label[split_data['test']['target_indices']]  # evaluation only

scale       : predictions=(145,), quantile crossings=0
calibration : predictions=(95,), quantile crossings=0
test        : predictions=(145,), quantile crossings=0


## 8. Raw Quantile Interval

In [38]:
def cqr_residual_score(y_true, q_low, q_high):
    return np.maximum(np.maximum(q_low - y_true, y_true - q_high), 0.0)

def conformal_quantile(scores, alpha=ALPHA):
    scores = np.asarray(scores)
    assert len(scores) == EXPECTED_CALIBRATION_SIZE
    assert np.isfinite(scores).all()
    rank = int(np.ceil((len(scores) + 1) * (1 - alpha)))
    assert rank == EXPECTED_CONFORMAL_RANK
    return np.sort(scores)[rank - 1], rank

def assert_valid_interval(lower, upper):
    assert lower.shape == upper.shape
    assert np.isfinite(lower).all() and np.isfinite(upper).all()
    assert np.all(lower <= upper)

raw_lower, raw_upper = test_q_low.copy(), test_q_high.copy()
assert_valid_interval(raw_lower, raw_upper)

## 9. Standard CQR

In [39]:
cal_standard_scores = cqr_residual_score(cal_y, cal_q_low, cal_q_high)
q_standard, standard_rank = conformal_quantile(cal_standard_scores)
standard_lower = test_q_low - q_standard
standard_upper = test_q_high + q_standard
assert_valid_interval(standard_lower, standard_upper)
print(f'Standard CQR: n_cal={len(cal_standard_scores)}, k={standard_rank}, q={q_standard:.6f}')

Standard CQR: n_cal=95, k=87, q=1.314278


## 10. Local-Variability Scale

In [40]:
def local_lag_scale(X):
    """Population standard deviation (ddof=0) of the five historical lag values.
    This convention is used identically for scale, calibration, and test.
    """
    return np.std(X, axis=1, ddof=0)

scale_local_scale = local_lag_scale(split_data['scale']['X'])
cal_local_scale = local_lag_scale(split_data['calibration']['X'])
test_local_scale = local_lag_scale(split_data['test']['X'])
scale_reference = np.median(scale_local_scale)

# Normalization is only a scale parameterization, not a separate method.
cal_local_scale_normalized = cal_local_scale / scale_reference
test_local_scale_normalized = test_local_scale / scale_reference

assert np.isfinite(scale_reference) and scale_reference > 0
assert np.all(cal_local_scale > 0) and np.all(test_local_scale > 0)
assert np.all(cal_local_scale_normalized > 0) and np.all(test_local_scale_normalized > 0)
print(f'Local-scale convention: np.std(ddof=0); scale-split median reference={scale_reference:.6f}.')

Local-scale convention: np.std(ddof=0); scale-split median reference=0.279260.


## 11. Local-Variability Scaled Additive CQR

In [41]:
# No true sigma or regime labels are used in this non-oracle method.
cal_additive_scores = cqr_residual_score(cal_y, cal_q_low, cal_q_high) / cal_local_scale_normalized
q_additive, additive_rank = conformal_quantile(cal_additive_scores)
local_additive_lower = test_q_low - q_additive * test_local_scale_normalized
local_additive_upper = test_q_high + q_additive * test_local_scale_normalized
assert additive_rank == EXPECTED_CONFORMAL_RANK
assert_valid_interval(local_additive_lower, local_additive_upper)
print(f'Local-Variability Scaled Additive CQR: k={additive_rank}, q={q_additive:.6f}')

Local-Variability Scaled Additive CQR: k=87, q=0.975892


## 12. Local-Variability Replacement CQR

In [42]:
# Separate center-based replacement geometry; no true sigma or regime labels are used.
cal_center = (cal_q_low + cal_q_high) / 2
test_center = (test_q_low + test_q_high) / 2
cal_replacement_scores = np.abs(cal_y - cal_center) / cal_local_scale_normalized
q_replacement, replacement_rank = conformal_quantile(cal_replacement_scores)
local_replacement_lower = test_center - q_replacement * test_local_scale_normalized
local_replacement_upper = test_center + q_replacement * test_local_scale_normalized
assert replacement_rank == EXPECTED_CONFORMAL_RANK
assert_valid_interval(local_replacement_lower, local_replacement_upper)
print(f'Local-Variability Replacement CQR: k={replacement_rank}, q={q_replacement:.6f}')

Local-Variability Replacement CQR: k=87, q=1.626742


## 13. Oracle Scaled-Additive CQR

In [43]:
# ORACLE ONLY: true sigma enters here and nowhere in the non-oracle methods.
cal_oracle_additive_scores = cqr_residual_score(cal_y, cal_q_low, cal_q_high) / cal_sigma
q_oracle_additive, oracle_additive_rank = conformal_quantile(cal_oracle_additive_scores)
oracle_additive_lower = test_q_low - q_oracle_additive * test_sigma
oracle_additive_upper = test_q_high + q_oracle_additive * test_sigma
assert oracle_additive_rank == EXPECTED_CONFORMAL_RANK
assert_valid_interval(oracle_additive_lower, oracle_additive_upper)

## 14. Oracle Replacement CQR

In [44]:
# ORACLE ONLY: true sigma enters here.
cal_oracle_replacement_scores = np.abs(cal_y - cal_center) / cal_sigma
q_oracle_replacement, oracle_replacement_rank = conformal_quantile(cal_oracle_replacement_scores)
oracle_replacement_lower = test_center - q_oracle_replacement * test_sigma
oracle_replacement_upper = test_center + q_oracle_replacement * test_sigma
assert oracle_replacement_rank == EXPECTED_CONFORMAL_RANK
assert_valid_interval(oracle_replacement_lower, oracle_replacement_upper)

## 15. Evaluation Table

In [45]:
def evaluate_interval(name, y_true, lower, upper, regimes):
    assert_valid_interval(lower, upper)
    covered = (y_true >= lower) & (y_true <= upper)
    widths = upper - lower
    low = regimes == 'low'
    high = regimes == 'high'
    assert low.any() and high.any()
    return {
        'method': name,
        'coverage': covered.mean(),
        'coverage_error': covered.mean() - (1 - ALPHA),
        'mean_width': widths.mean(),
        'median_width': np.median(widths),
        'low_coverage': covered[low].mean(),
        'high_coverage': covered[high].mean(),
        'low_mean_width': widths[low].mean(),
        'high_mean_width': widths[high].mean(),
        'high_low_width_ratio': widths[high].mean() / widths[low].mean(),
        'mean_width_over_oracle_high_low_scale_ratio': widths.mean() / ORACLE_HIGH_LOW_SCALE_RATIO,
    }

intervals = {
    'Raw Quantile': (raw_lower, raw_upper),
    'Standard CQR': (standard_lower, standard_upper),
    'Local-Variability Scaled Additive CQR': (local_additive_lower, local_additive_upper),
    'Local-Variability Replacement CQR': (local_replacement_lower, local_replacement_upper),
    'ORACLE Scaled-Additive CQR': (oracle_additive_lower, oracle_additive_upper),
    'ORACLE Replacement CQR': (oracle_replacement_lower, oracle_replacement_upper),
}
results = pd.DataFrame([evaluate_interval(name, test_y, *interval, test_regime) for name, interval in intervals.items()])
pd.set_option('display.precision', 6)
print(results.to_string(index=False))

                               method  coverage  coverage_error  mean_width  median_width  low_coverage  high_coverage  low_mean_width  high_mean_width  high_low_width_ratio  mean_width_over_oracle_high_low_scale_ratio
                         Raw Quantile  0.262069       -0.637931    0.862893      0.866414      0.333333           0.23        0.874750         0.857557              0.980346                                     0.172579
                         Standard CQR  0.882759       -0.017241    3.491449      3.494970      1.000000           0.83        3.503306         3.486113              0.995092                                     0.698290
Local-Variability Scaled Additive CQR  0.931034        0.031034    5.200683      5.386919      0.866667           0.96        2.206495         6.548068              2.967633                                     1.040137
    Local-Variability Replacement CQR  0.937931        0.037931    7.230781      7.649563      0.844444           0.98      

## 16. Diagnostics

In [46]:
local_scale_sigma_correlation = np.corrcoef(test_local_scale, test_sigma)[0, 1]
print(f'Local-scale versus true-sigma Pearson correlation (diagnostic only): {local_scale_sigma_correlation:.6f}')
print('This correlation is not evidence of conformal validity. Width-versus-scale correlation is intentionally not used as evidence because interval width mechanically contains the scale.')

Local-scale versus true-sigma Pearson correlation (diagnostic only): 0.771920
This correlation is not evidence of conformal validity. Width-versus-scale correlation is intentionally not used as evidence because interval width mechanically contains the scale.


## 17. Sanity Checks

In [48]:
assert len(cal_y) == EXPECTED_CALIBRATION_SIZE
assert standard_rank == additive_rank == replacement_rank == oracle_additive_rank == oracle_replacement_rank == EXPECTED_CONFORMAL_RANK
assert np.max(split_data['calibration']['target_indices']) < np.min(split_data['test']['target_indices'])
numeric_results = results.drop(columns='method').to_numpy(dtype=float)
assert np.isfinite(numeric_results).all()
print('PASS: split sizes, prediction shapes, finite values, no crossings, positive scales, score lengths, k=87, interval ordering, and calibration/test target separation.')
print('PASS: non-oracle intervals use local lag scales only; true sigma is restricted to evaluation and explicitly labeled oracle cells.')

PASS: split sizes, prediction shapes, finite values, no crossings, positive scales, score lengths, k=87, interval ordering, and calibration/test target separation.
PASS: non-oracle intervals use local lag scales only; true sigma is restricted to evaluation and explicitly labeled oracle cells.


## 18. Final Experiment Summary

In [49]:
historical = pd.DataFrame([
    ('Raw Quantile', 0.262069, 0.862893, np.nan),
    ('Standard CQR', 0.882759, 3.491449, np.nan),
    ('Local-Variability Scaled Additive CQR', 0.931035, 5.200658, 2.967629),
    ('ORACLE Scaled-Additive CQR', 0.979310, 4.905553, 3.197120),
], columns=['method', 'historical_coverage', 'historical_mean_width', 'historical_high_low_width_ratio'])
comparison = historical.merge(results[['method', 'coverage', 'mean_width', 'high_low_width_ratio']], on='method', how='left')
print('Historical prototype comparison (comparison only; do not tune to match):')
print(comparison.to_string(index=False))
print('\nHistorical prototype results are preserved separately. The present notebook is a cleaned implementation and must be rerun before its numerical results are treated as independently verified.')
print('These are empirical chronological time-series coverage measurements, not theorem-level iid split-conformal guarantees.')

Historical prototype comparison (comparison only; do not tune to match):
                               method  historical_coverage  historical_mean_width  historical_high_low_width_ratio  coverage  mean_width  high_low_width_ratio
                         Raw Quantile             0.262069               0.862893                              NaN  0.262069    0.862893              0.980346
                         Standard CQR             0.882759               3.491449                              NaN  0.882759    3.491449              0.995092
Local-Variability Scaled Additive CQR             0.931035               5.200658                         2.967629  0.931034    5.200683              2.967633
           ORACLE Scaled-Additive CQR             0.979310               4.905553                         3.197120  0.979310    4.905573              3.197126

Historical prototype results are preserved separately. The present notebook is a cleaned implementation and must be rerun before it